# Step 5 — qualify ONTD stops via current night train stops

Filters `step4_MatchingONTDtoOSM.csv` down to the stops a night train calls at
today, using `B-o-T_DataBase_stop_times.csv` as the reference. This is Tier 1 of
the qualification design (README, Stage C) — the highest-confidence signal, so
a stop lost here is lost from the catalog.

The matcher is geography-first, name-second, as Stage B specifies; it
runs against step 4's output so ONTD's country and ids carry through to the
seed. (The strategy was first built as a standalone schedule→OSM matcher —
that notebook is retired; its one capability the join-based matcher lacks,
diagnosing *why* a schedule stop is unmatched, lives on as the coverage check
at the bottom of this notebook.) Per schedule stop:

1. **Geo candidates** — every step 4 stop within `MAX_MATCH_RADIUS_M` of *any*
   coordinate the schedule reports for the stop. Distances use the best report,
   because the export stamps one trip's coordinate onto unrelated stops
   (Amsterdam Centraal's sits on the whole Zürich corridor) and a mean would be
   dragged off. Candidates are scored 70 % name / 30 % distance and labelled
   `exact` / `geo_name` / `geo_only` / `ambiguous`.
2. **Name fallback** — when nothing is in range, or the geo winner's name is
   unrelated (usually a wrong schedule coordinate): exact, then `rapidfuzz`,
   within `NAME_FALLBACK_MAX_KM` → `name_only`. An exact name whose
   coordinates disagree by more is still kept and flagged
   `name_coords_conflict` (keep-it-in principle).

Names are normalised the same way on both sides — transliterated, parenthetical
qualifiers dropped, station abbreviations expanded (`Hbf` → `Hauptbahnhof`,
`Centraal` → `central`). The previous name-gated version had no expansion and
no geo pass, which silently dropped 267 of 610 schedule stops including nearly
every major German and Austrian hub.

Outputs (all in `data/`):

| File | What |
|---|---|
| `step5_JoinedNTStops.csv` | qualified stops, one row per ONTD stop, step 4 columns plus match metadata |
| `unmatched_stops.csv` | schedule stops with no accepted match |
| `step5_review_flagged.csv` | `geo_only`, `ambiguous`, `name_coords_conflict` — check these |
| `step5_coord_conflicts_report.csv` | schedule stops whose own coordinate reports disagree by more than GPS jitter |
| `step5_duplicate_matches_report.csv` | ONTD stops claimed by more than one schedule name (alternate spellings) |

In [1]:
import csv
import re
from collections import Counter, defaultdict

import numpy as np
from rapidfuzz import fuzz, process
from unidecode import unidecode

from data_sources import DATA_DIR, ensure_local

## Parameters

In [2]:
STEP4_PATH = ensure_local("step4_MatchingONTDtoOSM.csv")
STOP_TIMES_PATH = ensure_local("B-o-T_DataBase_stop_times.csv")

OUTPUT_PATH = DATA_DIR / "step5_JoinedNTStops.csv"
UNMATCHED_PATH = DATA_DIR / "unmatched_stops.csv"
REVIEW_PATH = DATA_DIR / "step5_review_flagged.csv"
COORD_CONFLICTS_PATH = DATA_DIR / "step5_coord_conflicts_report.csv"
DUPLICATES_PATH = DATA_DIR / "step5_duplicate_matches_report.csv"

MAX_MATCH_RADIUS_M = 1_500  # Stage B: start 500 m, widen to 1.5 km
NAME_FALLBACK_MAX_KM = 20.0
NAME_SIMILARITY_THRESHOLD = 85
NAME_UNRELATED_BELOW = 40  # geo winner with a name score under this → try the name path
# Ceiling on the kept-and-flagged name match. A same-named station a few tens of
# km away is a bad schedule coordinate; hundreds of km away it is a different
# station entirely (schedule "Sofia" once matched a Moldovan village 681 km from
# the Bulgarian capital, whose own coordinate was correct).
NAME_CONFLICT_MAX_KM = 100.0
COORD_CONFLICT_THRESHOLD_M = 100  # spread above this is a data conflict, not GPS jitter

## Helpers

`ABBREVIATIONS` is what the previous version lacked. Extend it when the
unmatched or review reports show a naming convention that isn't covered — the
unmatched list is the pipeline's own test, since every current night train stop
should match.

In [3]:
ABBREVIATIONS = {
    "hbf": "hauptbahnhof",
    "hb": "hauptbahnhof",
    "bf": "bahnhof",
    "bhf": "bahnhof",
    "hp": "haltepunkt",
    "hln": "hlavni nadrazi",
    "st": "sankt",
    "gl": "glowny",
    "c": "central",
    "cs": "central",
    "centraal": "central",
    "centrale": "central",
    "centralstation": "central",
    "centrum": "central",
}


def fix_mojibake(text):
    """Repair UTF-8 text that was mis-decoded as CP1252. Only attempted on
    non-ASCII text and discarded unless it decodes cleanly, so genuine
    single-byte accents fail the round-trip safely."""
    if not text or text.isascii():
        return text
    try:
        return text.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


def normalize(name: str) -> str:
    """Transliterate, drop parenthetical qualifiers, expand abbreviations."""
    if not name:
        return ""
    text = unidecode(fix_mojibake(name)).lower()
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text).strip()
    return " ".join(ABBREVIATIONS.get(word, word) for word in text.split())


def parse_float(value):
    if value is None:
        return None
    text = str(value).strip().replace(",", ".")
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def distance_km(lat1, lon1, lat2, lon2):
    """Equirectangular approximation — accurate enough at station scale."""
    r = 6371.0088
    lat1, lon1, lat2, lon2 = (np.radians(v) for v in (lat1, lon1, lat2, lon2))
    mean_lat = (lat1 + lat2) / 2
    dx = (lon2 - lon1) * np.cos(mean_lat)
    dy = lat2 - lat1
    return r * np.sqrt(dx**2 + dy**2)


def medoid_and_spread(coords):
    """One representative coordinate per stop (the report closest to all the
    others) plus the largest pairwise spread in metres. Robust to a single
    outlier report; the spread flags stops whose reports genuinely disagree."""
    unique = sorted(set(coords))
    if not unique:
        return None, None, 0.0
    if len(unique) == 1:
        return unique[0][0], unique[0][1], 0.0
    lat = np.array([c[0] for c in unique])
    lon = np.array([c[1] for c in unique])
    pairwise_m = (
        distance_km(lat[:, None], lon[:, None], lat[None, :], lon[None, :]) * 1000
    )
    index = int(pairwise_m.sum(axis=1).argmin())
    return float(lat[index]), float(lon[index]), float(pairwise_m.max())

## Load step 4 output

Step 4 can list several OSM candidates per ONTD stop; only the nearest is a
valid match.

In [4]:
with open(STEP4_PATH, encoding="utf-8-sig", newline="") as fh:
    step4_raw = list(csv.DictReader(fh))

best_by_ontd = {}
for row in step4_raw:
    distance = parse_float(row.get("distance_km"))
    distance = float("inf") if distance is None else distance
    current = best_by_ontd.get(row["ontd_id"])
    if current is None or distance < current[0]:
        best_by_ontd[row["ontd_id"]] = (distance, row)

# Night trains stop at railway stations, so metro/tram/bus objects never
# belong in the candidate pool — without this filter the matcher happily
# picked "Duisburg Hauptbahnhof U" (the U-Bahn entrance, urban_transit,
# 70 m from the schedule coordinate) over the mainline station 248 m away.
# "other" is excluded too: ferries qualify through step 6, not the schedule.
MAINLINE_MODES = {"heavy_rail", "mixed", "undecided", ""}
step4_rows = [
    row
    for _, row in best_by_ontd.values()
    if (row.get("osm_station_mode") or "") in MAINLINE_MODES
]
for row in step4_rows:
    row["_norm"] = normalize(row["ontd_name"])
    row["_lat"] = parse_float(row["ontd_lat"])
    row["_lon"] = parse_float(row["ontd_lon"])
    # Several ONTD rows can normalize to the same name because normalize()
    # strips parenthetical qualifiers — "Leipzig Hauptbahnhof" and
    # "Leipzig Hauptbahnhof (Tiefgleise)" collapse to one key. Remember which
    # rows carried no qualifier so ties can prefer the station over one of
    # its platform groups.
    row["_plain_name"] = "(" not in row["ontd_name"]

geo_rows = [r for r in step4_rows if r["_lat"] is not None and r["_lon"] is not None]
geo_lat = np.array([r["_lat"] for r in geo_rows])
geo_lon = np.array([r["_lon"] for r in geo_rows])

by_norm = defaultdict(list)
for row in step4_rows:
    by_norm[row["_norm"]].append(row)
# Plain names first within each collapsed group, so wherever selection falls
# back to list order or ties on distance, the station beats its platform group.
for rows_for_name in by_norm.values():
    rows_for_name.sort(key=lambda row: not row["_plain_name"])
unique_norms = [name for name in by_norm if name]

print(f"step4 rows: {len(step4_raw)} -> {len(step4_rows)} after nearest-per-stop")
print(
    f"with coordinates: {len(geo_rows)}, distinct normalized names: {len(unique_norms)}"
)

step4 rows: 48617 -> 44322 after nearest-per-stop
with coordinates: 44321, distinct normalized names: 43133


## Load and collapse the schedule stops

`stop_id` in the export holds a stop *name*, repeated once per trip. Every
reported coordinate is kept for matching; the medoid becomes the stop's
representative coordinate in the output.

In [5]:
schedule_stops = {}
with open(STOP_TIMES_PATH, encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        name = fix_mojibake((row.get("stop_id") or "").strip())
        if not name:
            continue
        entry = schedule_stops.setdefault(normalize(name), {"name": name, "coords": []})
        lat = parse_float(row.get("stop_lat"))
        lon = parse_float(row.get("stop_lon"))
        if lat is not None and lon is not None:
            entry["coords"].append((lat, lon))

for entry in schedule_stops.values():
    entry["lat"], entry["lon"], entry["spread_m"] = medoid_and_spread(entry["coords"])
    entry["n_reports"] = len(entry["coords"])
    entry["coords"] = sorted(set(entry["coords"]))

print(f"distinct schedule stops: {len(schedule_stops)}")
print(
    f"without any coordinate: {sum(1 for e in schedule_stops.values() if not e['coords'])}"
)

distinct schedule stops: 610
without any coordinate: 13


## Match

In [6]:
def confidence_label(distance_m, name_score):
    if distance_m <= 500 and name_score >= 85:
        return "exact"
    if distance_m <= 500 and name_score >= 60:
        return "geo_name"
    if distance_m <= 500:
        return "geo_only"
    return "ambiguous"


def name_hit(norm_name, coords):
    """Exact, then fuzzy, name match within NAME_FALLBACK_MAX_KM. An exact
    name whose coordinates disagree by more is returned flagged rather than
    dropped — in practice that is a bad coordinate in the schedule export."""

    def nearest(candidates):
        if not coords:
            return candidates[0], None
        best_row, best_distance = None, float("inf")
        for row in candidates:
            if row["_lat"] is None:
                continue
            distance = min(
                float(distance_km(lat, lon, row["_lat"], row["_lon"]))
                for lat, lon in coords
            )
            if distance < best_distance:
                best_row, best_distance = row, distance
        return best_row, (None if best_row is None else best_distance)

    exact_far = None
    if norm_name in by_norm:
        row, distance = nearest(by_norm[norm_name])
        if row is not None:
            if distance is None or distance <= NAME_FALLBACK_MAX_KM:
                return row, distance, 100.0, "name_only"
            if distance <= NAME_CONFLICT_MAX_KM:
                exact_far = (row, distance, 100.0, "name_coords_conflict")

    hit = process.extractOne(
        norm_name,
        unique_norms,
        scorer=fuzz.WRatio,
        score_cutoff=NAME_SIMILARITY_THRESHOLD,
    )
    if hit is not None:
        row, distance = nearest(by_norm[hit[0]])
        if row is not None and (distance is None or distance <= NAME_FALLBACK_MAX_KM):
            return row, distance, float(hit[1]), "name_only"

    return exact_far


def match_schedule_stop(norm_name, entry):
    """Return (step4_row, distance_km, name_score, label); row is None if unmatched."""
    coords = entry["coords"]

    if coords:
        nearest_m = np.full(len(geo_rows), np.inf)
        for lat, lon in coords:
            nearest_m = np.minimum(
                nearest_m, distance_km(lat, lon, geo_lat, geo_lon) * 1000
            )
        in_range = np.where(nearest_m <= MAX_MATCH_RADIUS_M)[0]
        if len(in_range):
            scored = []
            for index in in_range:
                row = geo_rows[index]
                name_score = fuzz.token_sort_ratio(norm_name, row["_norm"])
                # Both terms are on a 0–100 scale so the weights are comparable
                # regardless of MAX_MATCH_RADIUS_M.
                score = 0.7 * name_score + 0.3 * (
                    100 - nearest_m[index] / (MAX_MATCH_RADIUS_M / 100)
                )
                scored.append((score, name_score, float(nearest_m[index]), row))
            _, name_score, distance_m, row = max(scored, key=lambda item: item[0])

            # normalize() collapses "X" and "X (Tiefgleise)" to the same name,
            # and the platform-group node often sits a few metres closer to the
            # schedule coordinate than the station object. Equal name evidence
            # within the radius: the plain-named station wins.
            if not row["_plain_name"]:
                plain = [
                    item
                    for item in scored
                    if item[3]["_plain_name"] and item[1] == name_score
                ]
                if plain:
                    _, name_score, distance_m, row = max(
                        plain, key=lambda item: item[0]
                    )

            # An unrelated name at the schedule coordinate usually means the
            # coordinate is wrong, not the name — let a name hit win if there is one.
            if name_score < NAME_UNRELATED_BELOW:
                fallback = name_hit(norm_name, coords)
                if fallback is not None:
                    return fallback
            return (
                row,
                distance_m / 1000,
                float(name_score),
                confidence_label(distance_m, name_score),
            )

    fallback = name_hit(norm_name, coords)
    if fallback is not None:
        return fallback
    return None, None, None, "unmatched"


matched, unmatched, claims = {}, [], defaultdict(list)
for norm_name, entry in schedule_stops.items():
    row, distance, name_score, label = match_schedule_stop(norm_name, entry)
    if row is None:
        unmatched.append(entry)
        continue
    claims[row["ontd_id"]].append(entry["name"])
    previous = matched.get(row["ontd_id"])
    if previous is None or (distance or 0.0) < (previous["distance"] or 0.0):
        matched[row["ontd_id"]] = {
            "row": row,
            "entry": entry,
            "distance": distance,
            "name_score": name_score,
            "label": label,
        }

print(
    f"schedule stops matched: {len(schedule_stops) - len(unmatched)} / {len(schedule_stops)}"
)
print(f"unique ONTD stops kept: {len(matched)}")
print(f"unmatched: {len(unmatched)}")
print(Counter(m["label"] for m in matched.values()))

schedule stops matched: 592 / 610
unique ONTD stops kept: 584
unmatched: 18
Counter({'exact': 443, 'geo_name': 64, 'name_only': 27, 'geo_only': 25, 'ambiguous': 24, 'name_coords_conflict': 1})


## Write the outputs

In [7]:
def fmt(value, digits=3):
    return "" if value is None else round(value, digits)


step4_columns = [key for key in step4_rows[0] if not key.startswith("_")]
match_columns = [
    "schedule_name",
    "schedule_lat",
    "schedule_lon",
    "schedule_n_reports",
    "schedule_coord_spread_m",
    "schedule_distance_km",
    "match_confidence",
    "name_score",
]

with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=step4_columns + match_columns)
    writer.writeheader()
    for match in matched.values():
        entry = match["entry"]
        record = {key: match["row"][key] for key in step4_columns}
        record.update(
            schedule_name=entry["name"],
            schedule_lat=fmt(entry["lat"], 7),
            schedule_lon=fmt(entry["lon"], 7),
            schedule_n_reports=entry["n_reports"],
            schedule_coord_spread_m=fmt(entry["spread_m"], 1),
            schedule_distance_km=fmt(match["distance"]),
            match_confidence=match["label"],
            name_score=fmt(match["name_score"], 0),
        )
        writer.writerow(record)

with open(UNMATCHED_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh, fieldnames=["stop_name", "stop_lat", "stop_lon", "n_reports"]
    )
    writer.writeheader()
    for entry in sorted(unmatched, key=lambda e: e["name"]):
        writer.writerow(
            {
                "stop_name": entry["name"],
                "stop_lat": fmt(entry["lat"], 7),
                "stop_lon": fmt(entry["lon"], 7),
                "n_reports": entry["n_reports"],
            }
        )

review_labels = {"geo_only", "ambiguous", "name_coords_conflict"}
flagged = sorted(
    (m for m in matched.values() if m["label"] in review_labels),
    key=lambda m: -(m["distance"] or 0.0),
)
with open(REVIEW_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "schedule_name",
            "ontd_name",
            "ontd_country",
            "osm_stop_id",
            "match_confidence",
            "name_score",
            "distance_km",
        ],
    )
    writer.writeheader()
    for match in flagged:
        writer.writerow(
            {
                "schedule_name": match["entry"]["name"],
                "ontd_name": match["row"]["ontd_name"],
                "ontd_country": match["row"]["ontd_country"],
                "osm_stop_id": match["row"]["osm_stop_id"],
                "match_confidence": match["label"],
                "name_score": fmt(match["name_score"], 0),
                "distance_km": fmt(match["distance"]),
            }
        )

conflicts = sorted(
    (e for e in schedule_stops.values() if e["spread_m"] > COORD_CONFLICT_THRESHOLD_M),
    key=lambda e: -e["spread_m"],
)
with open(COORD_CONFLICTS_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh, fieldnames=["stop_name", "n_reports", "coord_spread_m", "reported_coords"]
    )
    writer.writeheader()
    for entry in conflicts:
        writer.writerow(
            {
                "stop_name": entry["name"],
                "n_reports": entry["n_reports"],
                "coord_spread_m": fmt(entry["spread_m"], 1),
                "reported_coords": "; ".join(
                    f"{lat:.5f},{lon:.5f}" for lat, lon in entry["coords"]
                ),
            }
        )

duplicates = {ontd_id: names for ontd_id, names in claims.items() if len(names) > 1}
with open(DUPLICATES_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=["ontd_id", "ontd_name", "schedule_names"])
    writer.writeheader()
    for ontd_id, names in sorted(duplicates.items(), key=lambda kv: kv[1]):
        writer.writerow(
            {
                "ontd_id": ontd_id,
                "ontd_name": matched[ontd_id]["row"]["ontd_name"],
                "schedule_names": " | ".join(sorted(names)),
            }
        )

print(f"{OUTPUT_PATH.name}: {len(matched)} rows")
print(f"{UNMATCHED_PATH.name}: {len(unmatched)} rows")
print(f"{REVIEW_PATH.name}: {len(flagged)} rows")
print(f"{COORD_CONFLICTS_PATH.name}: {len(conflicts)} rows")
print(f"{DUPLICATES_PATH.name}: {len(duplicates)} rows")

step5_JoinedNTStops.csv: 584 rows
unmatched_stops.csv: 18 rows
step5_review_flagged.csv: 50 rows
step5_coord_conflicts_report.csv: 11 rows
step5_duplicate_matches_report.csv: 8 rows


## Review

Every current night train stop should match, so the unmatched list is the
pipeline's own test. The residual is expected to be stations genuinely absent
from ONTD (planned Rail Baltica stops, some eastern European stations) —
anything else means the abbreviation map or a threshold needs extending. Then
work through `step5_review_flagged.csv`, largest distance first.

In [8]:
for entry in sorted(unmatched, key=lambda e: e["name"]):
    print(f"  {entry['name']}")

  Briançon
  Burgas
  Dej
  Dimitrovgrad
  Härnösand
  Hässleholm C
  Iași
  Kryvyi Rih
  Mykolaiv
  Poltava
  Pärnu International (Rail Baltica)
  Roma Ostiense
  Rīga Airport (Rail Baltica)
  Sighișoara
  Silistra
  Sliven
  Veliko Tarnovo
  Åre


## Coverage check — why is each unmatched stop unmatched?

The matcher above can only qualify a schedule stop if **ONTD lists the
station**: candidates come from the step 4 join. So an unmatched stop has one
of two very different causes, and they need different fixes:

- **no OSM station nearby** — the schedule coordinate is wrong, or the station
  genuinely is not in OSM (planned Rail Baltica stops);
- **an OSM station is right there, but it never came through step 4** — ONTD
  does not list it, or step 4 matched that ONTD row to a different object.
  These are the catalog's silent ONTD-coverage debt: today they have to be
  hand-carried through step 6 (Poltava, Burgas...), and this check is what
  finds them systematically instead of one at a time.

Searches the unmatched stops directly against **all** step 3b stations
(mainline modes only) with the same geo-first scoring the matcher uses, and
writes `data/step5_ontd_coverage_gaps.csv`.


In [9]:
COVERAGE_PATH = DATA_DIR / "step5_ontd_coverage_gaps.csv"

step3b_stations = []
with open(
    ensure_local("step3b_output_osm_stations_classified.csv"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        if row["station_mode"] not in ("heavy_rail", "mixed", "undecided"):
            continue
        try:
            lat, lon = float(row["stop_lat"]), float(row["stop_lon"])
        except ValueError:
            continue
        step3b_stations.append(
            {
                "stop_id": row["stop_id"],
                "name": row["stop_name"],
                "mode": row["station_mode"],
                "lat": lat,
                "lon": lon,
            }
        )

# Which OSM objects step 4 already accounts for — an unmatched schedule stop
# whose nearby station is absent from this set is an ONTD coverage gap.
step4_osm_ids = {row["osm_stop_id"] for row in step4_rows if row["osm_stop_id"]}

gap_rows = []
for entry in unmatched:
    if entry["lat"] is None or entry["lon"] is None:
        gap_rows.append(
            {
                "schedule_name": entry["name"],
                "diagnosis": "no_coordinates",
                "osm_stop_id": "",
                "osm_name": "",
                "distance_km": "",
                "name_score": "",
            }
        )
        continue
    candidates = [
        (distance_km(entry["lat"], entry["lon"], s["lat"], s["lon"]), s)
        for s in step3b_stations
    ]
    near = [(d, s) for d, s in candidates if d <= MAX_MATCH_RADIUS_M / 1000]
    if not near:
        gap_rows.append(
            {
                "schedule_name": entry["name"],
                "diagnosis": "no_osm_station_within_radius",
                "osm_stop_id": "",
                "osm_name": "",
                "distance_km": "",
                "name_score": "",
            }
        )
        continue
    # Same weighting as the matcher: name similarity dominates, distance breaks ties.
    scored = sorted(
        near,
        key=lambda pair: (
            -(
                0.7
                * fuzz.token_sort_ratio(
                    normalize(entry["name"]), normalize(pair[1]["name"])
                )
                + 0.3 * (100 - pair[0] * 1000 / (MAX_MATCH_RADIUS_M / 100))
            )
        ),
    )
    d, best = scored[0]
    diagnosis = (
        "osm_object_matched_to_other_ontd_row"
        if best["stop_id"] in step4_osm_ids
        else "station_absent_from_ontd"
    )
    gap_rows.append(
        {
            "schedule_name": entry["name"],
            "diagnosis": diagnosis,
            "osm_stop_id": best["stop_id"],
            "osm_name": best["name"],
            "distance_km": f"{d:.3f}",
            "name_score": f"{fuzz.token_sort_ratio(normalize(entry['name']), normalize(best['name'])):.0f}",
        }
    )

with open(COVERAGE_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "schedule_name",
            "diagnosis",
            "osm_stop_id",
            "osm_name",
            "distance_km",
            "name_score",
        ],
    )
    writer.writeheader()
    writer.writerows(gap_rows)

print(f"{len(gap_rows)} unmatched stops diagnosed -> {COVERAGE_PATH.name}")
print(Counter(r["diagnosis"] for r in gap_rows))
for r in gap_rows:
    print(
        f"  {r['schedule_name'][:34]:36} {r['diagnosis']:36} "
        f"{r['osm_name'][:28]:30} {r['osm_stop_id']}"
    )

18 unmatched stops diagnosed -> step5_ontd_coverage_gaps.csv
Counter({'station_absent_from_ontd': 12, 'no_osm_station_within_radius': 6})
  Sliven                               station_absent_from_ontd             Сливен                         osm:n14055863804
  Burgas                               no_osm_station_within_radius                                        
  Dimitrovgrad                         station_absent_from_ontd             Димитровград                   osm:w529540422
  Silistra                             station_absent_from_ontd             Силистра                       osm:w421067799
  Sighișoara                           station_absent_from_ontd             Sighișoara                     osm:w1152498013
  Iași                                 station_absent_from_ontd             Eurovoyage (coach rental)      osm:n11895142291
  Dej                                  station_absent_from_ontd             Dej Triaj                      osm:w272187224
  Roma Ostiense  